# Train, verify and publish the BESSTIE RoBERTa model

This notebook retrains the project's pooled RoBERTa-base model with the original recipe, keeps it
this time, proves the service reproduces its test metrics, and uploads it to the Hugging Face Hub.

**Before you start (one-time, about 5 minutes)**

1. **GPU:** *Runtime → Change runtime type → T4 GPU → Save.*
2. **Hugging Face token:** at <https://huggingface.co/settings/tokens> create a token with **Write**
   access. In Colab, click the **key icon** in the left sidebar (*Secrets*) → *Add new secret* →
   name `HF_TOKEN`, paste the token, and switch **Notebook access** on.
3. **Dataset access:** open <https://huggingface.co/datasets/surrey-nlp/BESSTIE-CW-26> while logged
   in. If it asks you to accept conditions, accept them.
4. Have **`review-classifier-service.zip`** on your computer.

**Then run the cells in order** (*Shift+Enter*). Step 2 asks you to pick the zip file. Training
takes roughly 5–15 minutes per task on a T4; everything else is a few minutes.

Colab deletes files when the runtime disconnects. The model is safe once step 8 has uploaded it.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU found. Runtime > Change runtime type > T4 GPU > Save, then run this cell again."
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the code

Click **Choose files** and pick `review-classifier-service.zip`. If you have pushed the repository to
GitHub, put its URL in `REPO_URL` instead and nothing is uploaded.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import zipfile

# Once the repository is public, set REPO_URL to skip the upload:
# "https://github.com/mohammadabdalaziz241/review-classifier-service.git"
REPO_URL = ""
WORKDIR = "/content"

os.chdir(WORKDIR)
shutil.rmtree("review-classifier-service", ignore_errors=True)
if REPO_URL:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, "review-classifier-service"], check=True
    )
else:
    from google.colab import files

    uploaded = files.upload()  # pick review-classifier-service.zip
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(".")

assert pathlib.Path("review-classifier-service/pyproject.toml").exists(), (
    "No review-classifier-service/pyproject.toml found. Upload the zip exactly as you received it."
)
os.chdir(f"{WORKDIR}/review-classifier-service")
print("Code ready in", os.getcwd())

## 3. Install

In [ ]:
%pip install -q -e ".[hf,train]"

import sys

sys.path.insert(0, "src")  # lets this notebook import the package without restarting
print("Installed.")

## 4. Log in to Hugging Face

Reads the `HF_TOKEN` secret, checks it can upload, and checks you can read the dataset.

In [ ]:
from huggingface_hub import HfApi

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add your Hugging Face token as a Colab secret named HF_TOKEN (key icon in the left "
            "sidebar) and switch 'Notebook access' on. Then run this cell again."
        ) from exc

api = HfApi()
me = api.whoami()
HF_USER = me["name"]
role = me.get("auth", {}).get("accessToken", {}).get("role")
print("Logged in as", HF_USER)
if role == "read":
    print("WARNING: this token is read-only, so step 8 cannot upload. Create a Write token.")

DATASET = "surrey-nlp/BESSTIE-CW-26"
try:
    print(f"Dataset {DATASET} is readable (commit {api.dataset_info(DATASET).sha[:12]})")
except Exception as exc:
    raise RuntimeError(
        f"Cannot read {DATASET}. Open https://huggingface.co/datasets/{DATASET} while logged in, "
        "accept the access conditions if asked, then run this cell again."
    ) from exc

## 5. Choose what to train

Start with sentiment. For the sarcasm model afterwards, set `TASK = "sarcasm"` and run from this
cell down again. Seed 42 matches the notebook's first run.

In [ ]:
TASK = "sentiment"  # "sentiment" or "sarcasm"
SEED = 42
PUBLIC = False  # True makes the Hub repository public

OUTPUT = f"checkpoints/roberta-{TASK}-seed{SEED}"
REPO_ID = f"{HF_USER}/besstie-roberta-{TASK}"
os.makedirs("reports", exist_ok=True)
print(f"Will train {TASK} (seed {SEED}) into {OUTPUT} and upload to {REPO_ID}")

## 6. Train

Progress bars show each epoch. At the end it prints validation and test macro-F1.

In [ ]:
!python -m review_classifier.train --task {TASK} --seed {SEED} --output {OUTPUT} --dataset {DATASET} --overwrite

In [ ]:
import json

manifest = json.loads(pathlib.Path(OUTPUT, "training_manifest.json").read_text())
NOTEBOOK = {"sentiment": {42: 0.9028, 123: 0.9011}, "sarcasm": {42: 0.7001, 123: 0.7173}}

val, test = manifest["validation_metrics"], manifest["test_metrics"]
print(f"Dataset commit   {manifest['dataset']['version']}")
print(f"Trained on       {manifest['environment']['device']} in {manifest['train_seconds']:.0f}s")
print(f"Validation       macro-F1 {val['macro_f1']:.4f}  accuracy {val['accuracy']:.4f}")
print(
    f"Test             macro-F1 {test['macro_f1']:.4f}  accuracy {test['accuracy']:.4f}  "
    f"class-1 F1 {test['class_1_f1']:.4f}"
)
for variety, m in manifest["test_metrics_by_variety"].items():
    print(f"  {variety:7s}        macro-F1 {m['macro_f1']:.4f}  (n={m['n']})")
reference = NOTEBOOK[TASK].get(SEED)
if reference is not None:
    print(
        f"Original notebook, same seed: test macro-F1 {reference:.4f} "
        f"(difference {test['macro_f1'] - reference:+.4f}; small differences are normal)"
    )

## 7. Serve it and check the service reproduces the test metrics

Starts the real service on this machine with the new checkpoint, sends all test texts through
the API, and compares with the training metrics. It should end with **`Result MATCH`**.

In [ ]:
import time
import urllib.request


def start_service(model_id, revision=None, port=8000):
    env = dict(os.environ, MODEL_ID=model_id, PORT=str(port), MAX_TEXT_CHARS="100000")
    if revision:
        env["MODEL_REVISION"] = revision
    log_path = f"reports/service-{port}.log"
    with open(log_path, "w") as log:  # the service keeps its own handle to the file
        process = subprocess.Popen(
            [sys.executable, "-m", "review_classifier"],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    deadline = time.time() + 900
    while time.time() < deadline:
        if process.poll() is not None:
            print(pathlib.Path(log_path).read_text()[-4000:])
            raise RuntimeError("The service stopped during startup; its log is printed above.")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/ready", timeout=2) as response:
                if response.status == 200:
                    print(f"Service ready on port {port}")
                    return process
        except OSError:
            time.sleep(2)
    process.terminate()
    raise TimeoutError(f"The service did not become ready; see {log_path}")


def evaluate_service(port, report_path):
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "review_classifier.evaluate",
            "--url",
            f"http://127.0.0.1:{port}",
            "--manifest",
            f"{OUTPUT}/training_manifest.json",
            "--report",
            report_path,
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode == 2:
        print(result.stderr[-4000:])
        raise RuntimeError("The evaluation could not run; the error is printed above.")
    return json.loads(pathlib.Path(report_path).read_text())


LOCAL_REPORT = f"reports/eval-{TASK}-seed{SEED}-local.json"
service = start_service(OUTPUT)
try:
    local_report = evaluate_service(8000, LOCAL_REPORT)
finally:
    service.terminate()
    service.wait()

## 8. Upload to the Hugging Face Hub

Uploads only if step 7 reported a match, then prints the exact settings to serve this model.

In [ ]:
from review_classifier.train import push_to_hub

if not local_report["passed"]:
    raise RuntimeError(
        "Not uploading: the served metrics did not match training. Download the files in step 10 "
        "and send them to Claude."
    )
COMMIT = push_to_hub(pathlib.Path(OUTPUT), REPO_ID, private=not PUBLIC)
print(f"Uploaded: https://huggingface.co/{REPO_ID}")
print(f"Serve it with: MODEL_ID={REPO_ID} MODEL_REVISION={COMMIT}")

## 9. Check the uploaded copy (optional, about 2 minutes)

Starts the service from the Hub at the exact commit just uploaded, the way a deployment will, and
runs the same check. This proves the published model is the one that was verified.

In [ ]:
HUB_REPORT = f"reports/eval-{TASK}-seed{SEED}-hub.json"
service = start_service(REPO_ID, revision=COMMIT, port=8001)
try:
    hub_report = evaluate_service(8001, HUB_REPORT)
finally:
    service.terminate()
    service.wait()
assert hub_report["model"]["version"] == COMMIT, "The service did not load the uploaded commit."
print(
    "Hub copy verified."
    if hub_report["passed"]
    else "Hub copy did NOT match; send the files to Claude."
)

## 10. Download the results

Downloads a small zip with the training manifest, model card and evaluation reports (no weights).
Send it to Claude to continue.

In [ ]:
bundle = f"results-{TASK}-seed{SEED}.zip"
with zipfile.ZipFile(bundle, "w") as archive:
    for name in ("training_manifest.json", "serving_config.json", "README.md"):
        archive.write(f"{OUTPUT}/{name}", f"{TASK}/{name}")
    for report in sorted(pathlib.Path("reports").glob(f"eval-{TASK}-seed{SEED}-*.json")):
        archive.write(report, f"{TASK}/{report.name}")
    if "COMMIT" in globals():
        archive.writestr(f"{TASK}/hub.txt", f"MODEL_ID={REPO_ID}\nMODEL_REVISION={COMMIT}\n")
print("Created", bundle)

try:
    from google.colab import files

    files.download(bundle)
except ImportError:
    pass

## Next: the sarcasm model

Go back to step 5, set `TASK = "sarcasm"`, and run steps 5–10 again. Steps 1–4 do not need
repeating while the runtime stays connected.